In [1]:
from PIL import Image
import os
import kagglehub
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm
import torch


/home/tuxenjoyer69/.local/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class imagen():
    def __init__(self, path):
        self.dir = path
        self.target = path.split("/")[-1].split(".")[-2]
        self.image = Image.open(path)

imagenes = []

archivos = os.listdir("./dataset/samples")
imagenes = [imagen(f"./dataset/samples/{archivo}") for archivo in archivos  if archivo.endswith("png")]

In [3]:
class CaptchaDataset(Dataset):
    def __init__(self, directory, files, transform=None, device="cuda"):
        self.directory = directory
        self.transform = transform
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.files = [
            file for file in files
            if file.endswith(".png")
        ]

        self.image = []
        
        for file in self.files:
            filename = file
            path = os.path.join(self.directory, filename)
            imagen = Image.open(path).convert("RGB")
            imagen = self.transform(imagen)
            imagen = imagen.to(self.device)
            self.image.append(imagen)

        target_not_normalizated = [ file.split(".")[-2] for file in self.files]

        chars = sorted(set("".join(target_not_normalizated)))
        self.char_to_idx = {char: i for i, char in enumerate(chars)}
        self.idx_to_char = {i: char for char, i in self.char_to_idx.items()}


        self.target = torch.tensor([
            [
                self.char_to_idx[caracter]
                for caracter in target_normalizated
            ] 
            for target_normalizated in target_not_normalizated
            ])
    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        return self.image[index], self.target[index]

In [4]:
from sklearn.model_selection import train_test_split
files = [
    f
    for f in os.listdir("./dataset/samples")
    if f.endswith(".png")
]

train_files, test_files = train_test_split(
    files,
    test_size=0.2,
    random_state=42
)
transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = CaptchaDataset(
    "./dataset/samples",
    train_files,
    transform=transform
)

test_dataset = CaptchaDataset(
    "./dataset/samples",
    test_files,
    transform=transform
)

In [5]:
valores = train_dataset.idx_to_char.values()
longitud = len(train_dataset.files[0][:-4])
dimensiones = Image.open(f"./dataset/samples/{train_dataset.files[0]}").size

In [6]:
import random
from captcha.image import ImageCaptcha
from tqdm import tqdm
def generar_imagenes(width, height, letras, path, cantidad):
    image = ImageCaptcha(width=width, height=height)

    for _ in tqdm(range(cantidad)):    
        letras = [random.choice(list(valores)) for _ in range(5)]
        palabra = "".join(letras)

        image.write(palabra, f"{path}/{palabra}.png")

In [7]:
generar_imagenes(280,90,valores, "./dataset/generated",5593)

100%|██████████| 5593/5593 [00:21<00:00, 260.89it/s]
